In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy import special
from ipywidgets import VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# ELLIPTIC / CAUER LOW-PASS FILTER DESIGN EXERCISE
#
# Specifications:
#
#       k2 = ωp / ωs = 0.95
#       Ap = 0.3 dB
#       As = 60 dB
#
# Procedure:
#
# 1. Calculate D and the elliptic measure k1.
# 2. Calculate k2' and the complete elliptic integrals K2 and K2'.
# 3. Calculate the elliptic nome q(k2).
# 4. Calculate the minimum order N.
# 5. Normalize so that sqrt(ωp ωs) = 1.
# 6. Calculate p0 and W.
# 7. Calculate Ωm and Vm.
# 8. Calculate transmission zeros and stable poles.
# 9. Construct H(s) from the calculated poles and zeros.
# 10. Construct H(jω), magnitude, phase and group delay.
#
# No pole, zero, gain, or polynomial coefficient from the printed solution
# is hard-coded.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# FILTER SPECIFICATIONS
# ==============================================================================

k2 = 0.95
Ap = 0.3
As = 60.0

# ==============================================================================
# STEP 1: ATTENUATION RATIO AND ELLIPTIC MEASURE k1
# ==============================================================================

D = (10.0**(As / 10.0) - 1.0) / (10.0**(Ap / 10.0) - 1.0)

k1 = 1.0 / np.sqrt(D)

k1p = np.sqrt(1.0 - k1**2)

# ==============================================================================
# STEP 2: k2 AND COMPLEMENTARY MODULUS
# ==============================================================================

k2p = np.sqrt(1.0 - k2**2)

# ==============================================================================
# STEP 3: COMPLETE ELLIPTIC INTEGRALS
#
# scipy.special.ellipk uses parameter m = k².
# ==============================================================================

K2 = special.ellipk(k2**2)

K2p = special.ellipk(k2p**2)

# ==============================================================================
# STEP 4: ELLIPTIC NOME
# ==============================================================================

q = np.exp(-np.pi * K2p / K2)

q0 = 0.5 * (1.0 - np.sqrt(k2p)) / (1.0 + np.sqrt(k2p))

q_approx = q0 + 2.0 * q0**5 + 15.0 * q0**9

# ==============================================================================
# STEP 5: MINIMUM FILTER ORDER
# ==============================================================================

N_exact = np.log(16.0 * D) / np.log(1.0 / q)

N = int(np.ceil(N_exact))

# ==============================================================================
# STEP 6: NORMALIZED PASSBAND AND STOPBAND FREQUENCIES
# ==============================================================================

wp = np.sqrt(k2)

ws = 1.0 / np.sqrt(k2)

wc = np.sqrt(wp * ws)

# ==============================================================================
# STEP 7: AUXILIARY LOGARITHMIC QUANTITY
# ==============================================================================

log_term = np.log((10.0**(Ap / 20.0) + 1.0) / (10.0**(Ap / 20.0) - 1.0))

u0 = K2 * log_term / (N * np.pi)

# ==============================================================================
# STEP 8: REAL PARAMETER p0
#
# sn(j u,k) = j sn(u,k') / cn(u,k')
#
# scipy.special.ellipj uses parameter m = k².
# ==============================================================================

m_complement = 1.0 - k2**2

sn_c, cn_c, dn_c, ph_c = special.ellipj(u0, m_complement)

p0 = -wp * sn_c / cn_c

# ==============================================================================
# STEP 9: AUXILIARY PARAMETER W
# ==============================================================================

W = np.sqrt((1.0 + p0**2 / wp**2) * (1.0 + p0**2 / ws**2))

# ==============================================================================
# STEP 10: Ωm AND Vm
# ==============================================================================

L = N // 2

Omega_values = []
V_values = []

for m_index in range(1, L + 1):

    argument = (2.0 * m_index - 1.0) * K2 / N

    sn_m, cn_m, dn_m, ph_m = special.ellipj(argument, k2**2)

    Omega_m = sn_m / ws

    V_m = np.sqrt((1.0 - wp**2 * Omega_m**2) * (1.0 - ws**2 * Omega_m**2))

    Omega_values.append(Omega_m)

    V_values.append(V_m)

Omega_values = np.array(Omega_values)

V_values = np.array(V_values)

# ==============================================================================
# STEP 11: STABLE POLES
# ==============================================================================

positive_imag_poles = []

for m_index in range(L):

    Omega_m = Omega_values[m_index]

    V_m = V_values[m_index]

    denominator_m = 1.0 + p0**2 * Omega_m**2

    sigma_m = p0 * V_m / denominator_m

    omega_m = wp * ws * Omega_m * W / denominator_m

    positive_imag_poles.append(sigma_m + 1j * omega_m)

positive_imag_poles = np.array(positive_imag_poles)

poles_numeric = []

for p_m in positive_imag_poles:

    poles_numeric.append(p_m)

    poles_numeric.append(np.conjugate(p_m))

poles_numeric = np.array(poles_numeric)

# ==============================================================================
# STEP 12: TRANSMISSION ZEROS
# ==============================================================================

positive_imag_zeros = 1j / Omega_values

zeros_numeric = []

for z_m in positive_imag_zeros:

    zeros_numeric.append(z_m)

    zeros_numeric.append(np.conjugate(z_m))

zeros_numeric = np.array(zeros_numeric)

# ==============================================================================
# SYMBOLIC VARIABLES
# ==============================================================================

s = sp.symbols('s', real=True)

omega = sp.symbols('omega', real=True)

I = sp.I

# ==============================================================================
# STEP 13: CONSTRUCT POLE AND ZERO POLYNOMIALS
# ==============================================================================

denominator_symbolic = sp.Integer(1)

for p_k in poles_numeric:

    p_symbolic = sp.Float(p_k.real, 16) + I * sp.Float(p_k.imag, 16)

    denominator_symbolic *= s - p_symbolic

denominator_symbolic = sp.expand(denominator_symbolic)

denominator_symbolic = sp.N(sp.re(denominator_symbolic), 12)

zero_polynomial_symbolic = sp.Integer(1)

for z_k in zeros_numeric:

    z_symbolic = sp.Float(z_k.real, 16) + I * sp.Float(z_k.imag, 16)

    zero_polynomial_symbolic *= s - z_symbolic

zero_polynomial_symbolic = sp.expand(zero_polynomial_symbolic)

zero_polynomial_symbolic = sp.N(sp.re(zero_polynomial_symbolic), 12)

# ==============================================================================
# STEP 14: NORMALIZATION CONSTANT
# ==============================================================================

H_DC = 10.0**(-Ap / 20.0)

H0_symbolic = sp.N(H_DC * denominator_symbolic.subs(s, 0) / zero_polynomial_symbolic.subs(s, 0), 12)

numerator_symbolic = sp.expand(H0_symbolic * zero_polynomial_symbolic)

H_s = sp.cancel(numerator_symbolic / denominator_symbolic)

# ==============================================================================
# STEP 15: POLE AND ZERO FACTORS FOR DISPLAY
# ==============================================================================

pole_factors = []

for p_m in positive_imag_poles:

    sigma_m = sp.Float(p_m.real, 16)

    omega_m = sp.Float(p_m.imag, 16)

    factor_m = sp.expand(s**2 - 2.0 * sigma_m * s + sigma_m**2 + omega_m**2)

    pole_factors.append(sp.N(factor_m, 10))

zero_factors = []

for z_m in positive_imag_zeros:

    omega_z = sp.Float(z_m.imag, 16)

    factor_m = sp.expand(s**2 + omega_z**2)

    zero_factors.append(sp.N(factor_m, 10))

# ==============================================================================
# STEP 16: FREQUENCY RESPONSE
# ==============================================================================

numerator_jw = sp.expand(numerator_symbolic.subs(s, I * omega))

denominator_jw = sp.expand(denominator_symbolic.subs(s, I * omega))

num_real = sp.N(sp.re(numerator_jw), 12)

num_imag = sp.N(sp.im(numerator_jw), 12)

den_real = sp.N(sp.re(denominator_jw), 12)

den_imag = sp.N(sp.im(denominator_jw), 12)

# ==============================================================================
# STEP 17: MAGNITUDE RESPONSE
# ==============================================================================

num_power = sp.expand(num_real**2 + num_imag**2)

den_power = sp.expand(den_real**2 + den_imag**2)

magnitude_squared_symbolic = sp.cancel(num_power / den_power)

magnitude_symbolic = sp.sqrt(magnitude_squared_symbolic)

# ==============================================================================
# STEP 18: GROUP DELAY
# ==============================================================================

num_real_derivative = sp.diff(num_real, omega)

num_imag_derivative = sp.diff(num_imag, omega)

den_real_derivative = sp.diff(den_real, omega)

den_imag_derivative = sp.diff(den_imag, omega)

numerator_phase_derivative = sp.cancel((num_real * num_imag_derivative - num_imag * num_real_derivative) / (num_real**2 + num_imag**2))

denominator_phase_derivative = sp.cancel((den_real * den_imag_derivative - den_imag * den_real_derivative) / (den_real**2 + den_imag**2))

group_delay_symbolic = sp.cancel(denominator_phase_derivative - numerator_phase_derivative)

# ==============================================================================
# NUMERICAL FUNCTIONS FOR PLOTTING
# ==============================================================================

num_real_function = sp.lambdify(omega, num_real, 'numpy')

num_imag_function = sp.lambdify(omega, num_imag, 'numpy')

den_real_function = sp.lambdify(omega, den_real, 'numpy')

den_imag_function = sp.lambdify(omega, den_imag, 'numpy')

group_delay_function = sp.lambdify(omega, group_delay_symbolic, 'numpy')

# ==============================================================================
# FREQUENCY AXIS
# ==============================================================================

omega_values = np.logspace(-2, 2, 6000)

# ==============================================================================
# NUMERICAL FREQUENCY RESPONSE
# ==============================================================================

num_real_values = np.asarray(num_real_function(omega_values), dtype=float)

num_imag_values = np.asarray(num_imag_function(omega_values), dtype=float)

den_real_values = np.asarray(den_real_function(omega_values), dtype=float)

den_imag_values = np.asarray(den_imag_function(omega_values), dtype=float)

if num_real_values.ndim == 0:

    num_real_values = np.full_like(omega_values, float(num_real_values))

if num_imag_values.ndim == 0:

    num_imag_values = np.full_like(omega_values, float(num_imag_values))

H_values = (num_real_values + 1j * num_imag_values) / (den_real_values + 1j * den_imag_values)

magnitude_values = np.abs(H_values)

phase_values = np.unwrap(np.angle(H_values))

phase_deg_values = np.rad2deg(phase_values)

# ==============================================================================
# GROUP-DELAY VALUES
# ==============================================================================

group_delay_values = np.asarray(group_delay_function(omega_values), dtype=float)

if group_delay_values.ndim == 0:

    group_delay_values = np.full_like(omega_values, float(group_delay_values))

group_delay_values[~np.isfinite(group_delay_values)] = np.nan

# ==============================================================================
# TEXT
# ==============================================================================

omega_parameter_text = '<br>'.join([f'Ω{k + 1} = {Omega_values[k]:.6f}, V{k + 1} = {V_values[k]:.6f}' for k in range(L)])

zero_text = '<br>'.join([f'z{k + 1}, z{k + 1}* = ± {abs(positive_imag_zeros[k].imag):.6f}j' for k in range(L)])

pole_text = '<br>'.join([f'p{k + 1}, p{k + 1}* = {positive_imag_poles[k].real:+.6f} ± {positive_imag_poles[k].imag:.6f}j' for k in range(L)])

zero_factor_text = ''

for index, factor_k in enumerate(zero_factors):

    zero_factor_text += f'Zero pair {index + 1}: <span style="color:#0066cc;">{sp.sstr(factor_k)}</span><br>'

pole_factor_text = ''

for index, factor_k in enumerate(pole_factors):

    pole_factor_text += f'Pole pair {index + 1}: <span style="color:#0066cc;">{sp.sstr(factor_k)}</span><br>'

# ==============================================================================
# DISPLAY EXPRESSIONS
# ==============================================================================

H0_display = float(H0_symbolic)

numerator_display = sp.N(numerator_symbolic, 7)

denominator_display = sp.N(denominator_symbolic, 7)

num_real_display = sp.N(num_real, 7)

num_imag_display = sp.N(num_imag, 7)

den_real_display = sp.N(den_real, 7)

den_imag_display = sp.N(den_imag, 7)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1320px;
    max-width:1320px;
    box-sizing:border-box;
">
<b>Elliptic / Cauer Filter Design Exercise</b><br>
Design a normalized elliptic low-pass filter with
k<sub>2</sub> = ω<sub>p</sub>/ω<sub>s</sub> = {k2:.2f},
A<sub>p</sub> = {Ap:.1f} dB and
A<sub>s</sub> = {As:.0f} dB.
<br>
<b>Purpose:</b>
Follow the analytical Cauer design procedure. The elliptic integrals, nome,
minimum order, auxiliary parameters, poles, transmission zeros and transfer
function coefficients are calculated from the theoretical equations; no
numerical result from the printed solution is hard-coded.
</div>
""", layout=Layout(width='1330px', max_width='1330px'))

# ==============================================================================
# INFORMATION PANEL - TWO COLUMNS
# ==============================================================================

info_html = HTML(f"""
<div style="
    display:grid;
    grid-template-columns:1fr 1fr;
    gap:10px;
    width:760px;
    font-size:12px;
    line-height:1.50;
    box-sizing:border-box;
    align-items:start;
">

<!-- ===================================================================== -->
<!-- LEFT COLUMN: STEPS 1-7                                               -->
<!-- ===================================================================== -->

<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:10px 11px;
    background:white;
    box-sizing:border-box;
">

<b>Step 1 — Specifications</b><br>
<span style="color:#0066cc;">
k₂ = {k2:.6f}, Ap = {Ap:.1f} dB, As = {As:.0f} dB
</span>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 2 — Elliptic measures</b><br>
D = <span style="color:#0066cc;">{D:.6f}</span><br>
k₁ = <span style="color:#0066cc;">{k1:.9f}</span><br>
k₂ = <span style="color:#0066cc;">{k2:.6f}</span><br>
k₂' = <span style="color:#0066cc;">{k2p:.6f}</span>
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 3 — Elliptic integrals and nome</b><br>
K₂ = <span style="color:#0066cc;">{K2:.6f}</span><br>
K₂' = <span style="color:#0066cc;">{K2p:.6f}</span><br>
q(k₂) = <span style="color:#0066cc;">{q:.6f}</span><br>
q₀ = <span style="color:#0066cc;">{q0:.6f}</span><br>
q approximation = <span style="color:#0066cc;">{q_approx:.6f}</span>
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 4 — Minimum filter order</b><br>
Nmin = <span style="color:#0066cc;">{N_exact:.6f}</span><br>
N = <span style="color:#0066cc;"><b>{N}</b></span>
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 5 — Normalization</b><br>
ωp = <span style="color:#0066cc;">{wp:.6f}</span><br>
ωs = <span style="color:#0066cc;">{ws:.6f}</span><br>
√(ωpωs) = <span style="color:#0066cc;">{wc:.6f}</span>
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 6 — Auxiliary quantities</b><br>
u₀ = <span style="color:#0066cc;">{u0:.6f}</span><br>
p₀ = <span style="color:#0066cc;">{p0:.6f}</span><br>
W = <span style="color:#0066cc;">{W:.6f}</span>
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 7 — Ωm and Vm</b><br>
<span style="color:#0066cc;">
{omega_parameter_text}
</span>
</div>

</div>

<!-- ===================================================================== -->
<!-- RIGHT COLUMN: STEPS 8-14                                             -->
<!-- ===================================================================== -->

<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:10px 11px;
    background:white;
    box-sizing:border-box;
">

<b>Step 8 — Transmission zeros</b><br>
<span style="color:#0066cc;">
{zero_text}
</span>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 9 — Stable poles</b><br>
<span style="color:#0066cc;">
{pole_text}
</span>
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 10 — Zero factors</b><br>
{zero_factor_text}
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 11 — Pole factors</b><br>
{pole_factor_text}
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 12 — Normalization constant</b><br>
H₀ = <span style="color:#0066cc;">{H0_display:.8f}</span>
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 13 — Transfer function</b><br>
N(s) =
<span style="color:#0066cc;">
{sp.sstr(numerator_display)}
</span><br><br>
D(s) =
<span style="color:#0066cc;">
{sp.sstr(denominator_display)}
</span>
</div>

<div style="margin-top:7px; padding-top:6px; border-top:1px solid #eeeeee;">
<b>Step 14 — Frequency response</b><br>
N(jω) =
<span style="color:#0066cc;">
({sp.sstr(num_real_display)}) + j({sp.sstr(num_imag_display)})
</span><br><br>
D(jω) =
<span style="color:#0066cc;">
({sp.sstr(den_real_display)}) + j({sp.sstr(den_imag_display)})
</span>
</div>

</div>

</div>
""", layout=Layout(width='770px', max_width='770px'))

# ==============================================================================
# COMMON FIGURE SETTINGS
# ==============================================================================

title_fontsize = 11
label_fontsize = 9
tick_fontsize = 8
legend_fontsize = 8

# ==============================================================================
# FIGURE 1: MAGNITUDE RESPONSE
# ==============================================================================

fig_mag, ax_mag = plt.subplots(figsize=(5.3, 3.0))

ax_mag.plot(omega_values, magnitude_values, 'r-', linewidth=2.0, label='|H(jω)|')

ax_mag.axvline(wp, color='black', linestyle=':', linewidth=1.0, label='ωp')

ax_mag.axvline(ws, color='gray', linestyle=':', linewidth=1.0, label='ωs')

ax_mag.axhline(10.0**(-Ap / 20.0), color='gray', linestyle='--', linewidth=0.9, label='Passband limit')

ax_mag.set_xscale('log')

ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)

ax_mag.set_ylabel('|H(jω)|', fontsize=label_fontsize)

ax_mag.set_title('Elliptic Filter Magnitude Response', fontsize=title_fontsize, fontweight='bold', pad=5)

ax_mag.tick_params(axis='both', labelsize=tick_fontsize)

ax_mag.grid(True, which='both', linestyle=':', alpha=0.5)

ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=4, fontsize=legend_fontsize)

ax_mag.set_xlim(0.01, 100.0)

ax_mag.set_ylim(0.0, 1.08)

fig_mag.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)

fig_mag.canvas.header_visible = False

fig_mag.canvas.toolbar_visible = False

fig_mag.canvas.resizable = False

fig_mag.canvas.layout.width = '530px'

fig_mag.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 2: PHASE RESPONSE
# ==============================================================================

fig_phase, ax_phase = plt.subplots(figsize=(5.3, 3.0))

ax_phase.plot(omega_values, phase_deg_values, 'r-', linewidth=2.0, label='∠H(jω)')

ax_phase.axvline(wp, color='black', linestyle=':', linewidth=1.0, label='ωp')

ax_phase.axvline(ws, color='gray', linestyle=':', linewidth=1.0, label='ωs')

ax_phase.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_phase.set_xscale('log')

ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)

ax_phase.set_ylabel('Phase (degrees)', fontsize=label_fontsize)

ax_phase.set_title('Elliptic Filter Phase Response', fontsize=title_fontsize, fontweight='bold', pad=5)

ax_phase.tick_params(axis='both', labelsize=tick_fontsize)

ax_phase.grid(True, which='both', linestyle=':', alpha=0.5)

ax_phase.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=3, fontsize=legend_fontsize)

ax_phase.set_xlim(0.01, 100.0)

fig_phase.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)

fig_phase.canvas.header_visible = False

fig_phase.canvas.toolbar_visible = False

fig_phase.canvas.resizable = False

fig_phase.canvas.layout.width = '530px'

fig_phase.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 3: GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(5.3, 3.0))

ax_gd.plot(omega_values, group_delay_values, 'r-', linewidth=2.0, label='τ(ω)')

ax_gd.axvline(wp, color='black', linestyle=':', linewidth=1.0, label='ωp')

ax_gd.axvline(ws, color='gray', linestyle=':', linewidth=1.0, label='ωs')

ax_gd.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_gd.set_xscale('log')

ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=label_fontsize)

ax_gd.set_ylabel('Group Delay τ(ω)', fontsize=label_fontsize)

ax_gd.set_title('Elliptic Filter Group Delay', fontsize=title_fontsize, fontweight='bold', pad=5)

ax_gd.tick_params(axis='both', labelsize=tick_fontsize)

ax_gd.grid(True, which='both', linestyle=':', alpha=0.5)

ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), ncol=3, fontsize=legend_fontsize)

ax_gd.set_xlim(0.01, 100.0)

ax_gd.set_ylim(-20.0, 20.0)

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.30, top=0.85)

fig_gd.canvas.header_visible = False

fig_gd.canvas.toolbar_visible = False

fig_gd.canvas.resizable = False

fig_gd.canvas.layout.width = '530px'

fig_gd.canvas.layout.height = '305px'

# ==============================================================================
# LAYOUT
# ==============================================================================

left_column = VBox([info_html], layout=Layout(width='780px', min_width='780px', max_width='780px', flex='0 0 780px', align_items='flex-start'))

right_column = VBox([fig_mag.canvas, fig_phase.canvas, fig_gd.canvas], layout=Layout(width='540px', min_width='540px', max_width='540px', flex='0 0 540px', align_items='flex-start'))

main_layout = HBox([left_column, right_column], layout=Layout(width='1330px', min_width='1330px', max_width='1330px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)